In [ ]:
import os
import subprocess
import pandas as pd
import matplotlib.pyplot as plt

os.makedirs("result", exist_ok=True)
print("Environment ready!")

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

print("Compiling C executables...")

exe_suffix = ".exe" if os.name == "nt" else ""
percolation_exe = Path(f"percolation_sim{exe_suffix}")
seq_exe = Path(f"seq_generator{exe_suffix}")

def run_checked(cmd):
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(
            "Command failed: " + " ".join(cmd) + "\n" +
            (result.stderr or result.stdout)
        )
    return result

if shutil.which("make"):
    subprocess.run(["make", "clean"], capture_output=True, text=True)
    run_checked(["make"])
else:
    source_files = [str(p) for p in Path("src").glob("*.c")]
    run_checked(["gcc", "-Wall", "-Wextra", "-O3", "-std=c99", "-o", str(percolation_exe), *source_files, "-lm"])
    run_checked(["gcc", "-Wall", "-Wextra", "-O3", "-std=c11", "-o", str(seq_exe), "seq_generator.c", "-lm"])

print(f"Compilation successful: {percolation_exe} and {seq_exe} are ready.")


In [ ]:
import os
import subprocess
from pathlib import Path

exe_suffix = ".exe" if os.name == "nt" else ""
percolation_exe = Path(f"percolation_sim{exe_suffix}")
run_cmd = [str(Path.cwd() / percolation_exe)]

print("Starting Hypergraph Percolation Simulation (this may take a few minutes)...\n")

process = subprocess.Popen(
    run_cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

is_progressing = False

for line in process.stdout:
    text = line.strip()
    if text:
        if text.startswith("Finished"):
            print(f"\r{text.ljust(60)}", end="", flush=True)
            is_progressing = True
        else:
            if is_progressing:
                print()
                is_progressing = False
            print(text)

process.wait()

if process.returncode == 0:
    print("\nSimulation completed successfully!")
else:
    print("\nSimulation aborted or encountered an error. Please check the C output above.")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import AutoMinorLocator
from matplotlib.lines import Line2D

plt.style.use('seaborn-v0_8-white')

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['font.size'] = 18
plt.rcParams['axes.labelsize'] = 28
plt.rcParams['xtick.labelsize'] = 20
plt.rcParams['ytick.labelsize'] = 20
plt.rcParams['legend.fontsize'] = 26
plt.rcParams['axes.linewidth'] = 1.5
plt.rcParams['xtick.major.width'] = 1.5
plt.rcParams['ytick.major.width'] = 1.5
plt.rcParams['xtick.minor.width'] = 1.0
plt.rcParams['ytick.minor.width'] = 1.0

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white"
})

prename = 'ph_data'
filename = f'result/{prename}.txt'

try:
    df = pd.read_csv(filename, delimiter=',')
    
    if 'node_survival' in df.columns:
        df.rename(columns={'node_survival': 'p_N'}, inplace=True)
    if 'hyperedge_survival' in df.columns:
        df.rename(columns={'hyperedge_survival': 'p_H'}, inplace=True)

    if 'p_N' in df.columns and 'p_H' in df.columns:
        if df['p_N'].nunique() > 1 and df['p_H'].nunique() > 1:
            x_col = 'p_N'
            x_label = '$p=p_N=p_H$'
            print("Detected sweep variable: Joint Percolation (sweeping both p_N and p_H)")
        elif df['p_N'].nunique() > 1:
            x_col = 'p_N'
            x_label = '$p_N$'
            print("Detected sweep variable: p_N (Node Percolation)")
        else:
            x_col = 'p_H'
            x_label = '$p_H$'
            print("Detected sweep variable: p_H (Hyperedge Percolation)")
    elif 'p_N' in df.columns:
        x_col = 'p_N'
        x_label = '$p_N$'
        print("Detected sweep variable: p_N")
    else:
        # Fallback
        x_col = 'p_H'
        x_label = '$p_H$'
        print("Detected sweep variable: p_H")

    p_values_to_show = np.arange(0.0, 1.01, 0.05)
    df_scatter = df[df[x_col].round(2).isin(p_values_to_show.round(2))]

    fig, ax = plt.subplots(figsize=(10, 8))


    ax.plot(df[x_col], df['mp_hwcc'], color='crimson',     linestyle='-', linewidth=3, zorder=2, clip_on=False)
    ax.plot(df[x_col], df['mp_gout'], color='dodgerblue',  linestyle='-', linewidth=3, zorder=2, clip_on=False)
    ax.plot(df[x_col], df['mp_gin'],  color='darkorange',  linestyle='-', linewidth=3, zorder=2, clip_on=False)
    ax.plot(df[x_col], df['mp_gscc'], color='forestgreen', linestyle='-', linewidth=3, zorder=2, clip_on=False)

    ax.scatter(df_scatter[x_col], df_scatter['sim_hwcc'], facecolors='none', edgecolors='crimson',     marker='D', s=100, linewidth=2, zorder=3, clip_on=False)
    ax.scatter(df_scatter[x_col], df_scatter['sim_gout'], facecolors='none', edgecolors='dodgerblue',  marker='o', s=100, linewidth=2, zorder=3, clip_on=False)
    ax.scatter(df_scatter[x_col], df_scatter['sim_gin'],  facecolors='none', edgecolors='darkorange',  marker='s', s=100, linewidth=2, zorder=3, clip_on=False)
    ax.scatter(df_scatter[x_col], df_scatter['sim_gscc'], facecolors='none', edgecolors='forestgreen', marker='^', s=100, linewidth=2, zorder=3, clip_on=False)

    ax.set_xlabel(x_label, fontstyle='italic')
    ax.set_ylabel(r'$R^{W}\mathrm{,}\,R^{(+)}\mathrm{,}\,R^{(-)}\mathrm{,}\,R$')

    ax.set_xlim(0.0, 1.0)
    ax.set_ylim(0, 1.0)
    ax.set_xticks([0.0, 0.2, 0.4, 0.6, 0.8, 1.0])
    ax.set_yticks([0.0, 0.2, 0.4, 0.6, 0.8, 1.0])
    ax.set_xticklabels(['', '0.2', '0.4', '0.6', '0.8', '1.0'])
    ax.set_yticklabels(['', '0.2', '0.4', '0.6', '0.8', '1.0'])
    ax.text(0, 0, '0', horizontalalignment='right', verticalalignment='top', transform=ax.transAxes)

    ax.xaxis.set_minor_locator(AutoMinorLocator(5))
    ax.yaxis.set_minor_locator(AutoMinorLocator(5))
    ax.tick_params(which='major', direction='in', length=8)
    ax.tick_params(which='minor', direction='in', length=4)

    line_handles = [
        Line2D([0], [0], color='crimson',     lw=2.5),
        Line2D([0], [0], color='dodgerblue',  lw=2.5),
        Line2D([0], [0], color='darkorange',  lw=2.5),
        Line2D([0], [0], color='forestgreen', lw=2.5)
    ]
    marker_handles = [
        Line2D([0], [0], color='crimson',     marker='D', linestyle='None', markersize=8, markerfacecolor='none', markeredgewidth=1.5),
        Line2D([0], [0], color='dodgerblue',  marker='o', linestyle='None', markersize=8, markerfacecolor='none', markeredgewidth=1.5),
        Line2D([0], [0], color='darkorange',  marker='s', linestyle='None', markersize=8, markerfacecolor='none', markeredgewidth=1.5),
        Line2D([0], [0], color='forestgreen', marker='^', linestyle='None', markersize=8, markerfacecolor='none', markeredgewidth=1.5)
    ]

    handles = line_handles + marker_handles
    labels = ['', '', '', '', 'HWCC', 'HGOUT', 'HGIN', 'HGSCC']

    ax.legend(handles, labels,
              loc='best',
              ncol=2,
              frameon=False,
              fontsize=28,
              handlelength=2.5,
              handletextpad=0.1,
              columnspacing=0.1)

    output_filename = f'fig/{prename}.pdf'
    plt.savefig(output_filename, bbox_inches='tight', dpi=600)

    print(f"File saved: {output_filename}")

except Exception as e:
    print(f"Error: {e}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
from matplotlib.lines import Line2D
import warnings

warnings.filterwarnings("ignore")

files = {
    'Poisson Uncorrelated': 'result/poisson_uncor.txt',
    'Poisson Correlated': 'result/poisson_cor.txt',
    'SF Uncorrelated': 'result/sf_uncor.txt',
    'SF Correlated': 'result/sf_cor.txt'
}

missing_files = [path for path in files.values() if not os.path.exists(path)]
if missing_files:
    print("Skipping topology comparison plot because these input files are missing:")
    for path in missing_files:
        print(f"  - {path}")
    print("Generate these result files first, then rerun this cell.")
else:
    data = {name: pd.read_csv(path) for name, path in files.items()}

    fig, axes = plt.subplots(2, 4, figsize=(24, 12), sharex=True, sharey=True)

    rows = ['Poisson Networks', r'Scale-Free Networks ($\gamma=2.5$)']
    topo_bases = ['Poisson', 'SF']
    components = ['hwcc', 'gin', 'gout', 'gscc']
    titles = ['HGWCC (Undirected)', 'HGIN', 'HGOUT', 'HGSCC (Directed Core)']

    for row_idx, topo_base in enumerate(topo_bases):
        df_uncor = data[f'{topo_base} Uncorrelated']
        df_cor = data[f'{topo_base} Correlated']

        for col_idx, comp in enumerate(components):
            ax = axes[row_idx, col_idx]

            uncor_N = df_uncor[df_uncor['type'] == 'N']
            uncor_H = df_uncor[df_uncor['type'] == 'H']
            cor_N = df_cor[df_cor['type'] == 'N']
            cor_H = df_cor[df_cor['type'] == 'H']

            marker_spacing = 5

            ax.plot(uncor_N['p'], uncor_N[f'mp_{comp}'], '--', color='tab:blue', linewidth=2.5, alpha=0.85)
            ax.plot(uncor_N['p'], uncor_N[f'sim_{comp}'], 's', color='tab:blue', mfc='none',
                    markersize=7, markeredgewidth=1.5, linestyle='none', markevery=marker_spacing)

            ax.plot(uncor_H['p'], uncor_H[f'mp_{comp}'], '--', color='tab:red', linewidth=2.5, alpha=0.85)
            ax.plot(uncor_H['p'], uncor_H[f'sim_{comp}'], '^', color='tab:red', mfc='none',
                    markersize=8, markeredgewidth=1.5, linestyle='none', markevery=marker_spacing)

            ax.plot(cor_N['p'], cor_N[f'mp_{comp}'], '-', color='tab:blue', linewidth=2.5, alpha=0.85)
            ax.plot(cor_N['p'], cor_N[f'sim_{comp}'], 's', color='tab:blue', mfc='tab:blue',
                    markersize=7, markeredgewidth=1.5, linestyle='none', markevery=marker_spacing)

            ax.plot(cor_H['p'], cor_H[f'mp_{comp}'], '-', color='tab:red', linewidth=2.5, alpha=0.85)
            ax.plot(cor_H['p'], cor_H[f'sim_{comp}'], '^', color='tab:red', mfc='tab:red',
                    markersize=8, markeredgewidth=1.5, linestyle='none', markevery=marker_spacing)

            if row_idx == 0:
                ax.set_title(titles[col_idx], fontsize=18, fontweight='bold', pad=15)

            ax.grid(True, linestyle=':', alpha=0.7)
            ax.set_ylim(-0.05, 1.05)
            ax.set_xlim(-0.02, 1.02)
            ax.tick_params(axis='both', which='major', labelsize=12)

            if row_idx == 1:
                ax.set_xlabel(r'Survival Probability ($p_N$ or $p_H$)', fontsize=16)

            if col_idx == 0:
                ax.set_ylabel(r'Normalized Component Size', fontsize=16)
                ax.annotate(rows[row_idx], xy=(-0.35, 0.5), xycoords='axes fraction',
                            fontsize=20, fontweight='bold', ha='center', va='center', rotation=90)

            if row_idx == 0 and col_idx == 0:
                custom_lines = [
                    Line2D([0], [0], color='tab:blue', marker='s', mfc='none', markersize=8, markeredgewidth=1.5, linestyle='--', lw=2.5),
                    Line2D([0], [0], color='tab:red', marker='^', mfc='none', markersize=9, markeredgewidth=1.5, linestyle='--', lw=2.5),
                    Line2D([0], [0], color='tab:blue', marker='s', mfc='tab:blue', markersize=8, markeredgewidth=1.5, linestyle='-', lw=2.5),
                    Line2D([0], [0], color='tab:red', marker='^', mfc='tab:red', markersize=9, markeredgewidth=1.5, linestyle='-', lw=2.5)
                ]
                ax.legend(custom_lines,
                          [r'Uncorrelated: Node ($p_N$)',
                           r'Uncorrelated: Hyperedge ($p_H$)',
                           r'Correlated: Node ($p_N$)',
                           r'Correlated: Hyperedge ($p_H$)'],
                          fontsize=13, loc='upper left', framealpha=0.9)

    plt.tight_layout()
    plt.subplots_adjust(left=0.08, wspace=0.08, hspace=0.15)

    os.makedirs('fig', exist_ok=True)
    output_fig = 'fig/Figure1(1)_giant_component.pdf'
    plt.savefig(output_fig, bbox_inches='tight', dpi=600)
    print(f"Figure saved: {output_fig}")

    plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq
import warnings
import os
import subprocess
from pathlib import Path

warnings.filterwarnings("ignore")

exe_suffix = ".exe" if os.name == "nt" else ""
seq_exe = Path(f"seq_generator{exe_suffix}")
seq_cmd = [str(Path.cwd() / seq_exe)]
N_SEQUENCE_SAMPLE = 2000

if not seq_exe.exists():
    compile_result = subprocess.run(
        ["gcc", "-Wall", "-Wextra", "-O3", "-std=c11", "-o", str(seq_exe), "seq_generator.c", "-lm"],
        capture_output=True,
        text=True
    )
    if compile_result.returncode != 0:
        raise RuntimeError("Failed to compile seq_generator:\n" + (compile_result.stderr or compile_result.stdout))

def generate_sequences_from_c(topology_type, lam=3.0, gamma=3.5):
    topo_map = {
        'Uncorrelated Poisson': 0,
        'Correlated Poisson': 1,
        'Uncorrelated SF': 2,
        'Correlated SF': 3
    }
    topo_int = topo_map[topology_type]

    try:
        result = subprocess.check_output(
            [*seq_cmd, str(topo_int), str(lam), str(gamma), str(N_SEQUENCE_SAMPLE)],
            text=True
        )
    except FileNotFoundError:
        raise RuntimeError("seq_generator was not found. Compile it before running this cell.")
    except subprocess.CalledProcessError as e:
        raise RuntimeError(f"seq_generator failed with exit code {e.returncode}.")

    lines = result.strip().split('\n')
    N, Q = map(int, lines[0].split())

    k_in = np.zeros(N, dtype=int)
    k_out = np.zeros(N, dtype=int)
    for i in range(N):
        k_in[i], k_out[i] = map(int, lines[1 + i].split())

    size_T = np.zeros(Q, dtype=int)
    size_H = np.zeros(Q, dtype=int)
    for i in range(Q):
        size_T[i], size_H[i] = map(int, lines[1 + N + i].split())

    return k_in, k_out, size_T, size_H

def find_thresholds(theta, q_in, q_out, m_in, m_out):
    q = q_in + q_out
    m = m_in + m_out

    mean_q_out = np.mean(q_out)
    mean_m_out = np.mean(m_out)
    mean_q = np.mean(q)
    mean_m = np.mean(m)

    B_dir = np.mean(q_in * q_out) / mean_q_out
    B_undir = np.mean(q * (q - 1)) / mean_q

    C_dir = m_in * m_out
    sum_C_dir_by_m = np.bincount(m, weights=C_dir)
    valid_m_dir = np.nonzero(sum_C_dir_by_m)[0]
    valid_sum_C_dir = sum_C_dir_by_m[valid_m_dir]

    C_undir = m * (m - 1)
    sum_C_undir_by_m = np.bincount(m, weights=C_undir)
    valid_m_undir = np.nonzero(sum_C_undir_by_m)[0]
    valid_sum_C_undir = sum_C_undir_by_m[valid_m_undir]

    Q_total = len(m)

    def dir_func(p):
        pi_N = 1.0 - theta + theta * p
        pi_N = max(pi_N, 1e-12)
        H_term = np.sum((pi_N ** (valid_m_dir - 2)) * valid_sum_C_dir) / Q_total / mean_m_out
        return p * B_dir * H_term - 1.0

    def undir_func(p):
        pi_N = 1.0 - theta + theta * p
        pi_N = max(pi_N, 1e-12)
        H_term = np.sum((pi_N ** (valid_m_undir - 2)) * valid_sum_C_undir) / Q_total / mean_m
        return p * B_undir * H_term - 1.0

    try:
        p_c_dir = brentq(dir_func, 1e-6, 1.0)
    except ValueError:
        p_c_dir = np.nan

    try:
        p_c_undir = brentq(undir_func, 1e-6, 1.0)
    except ValueError:
        p_c_undir = np.nan

    return p_c_dir, p_c_undir

thetas = np.linspace(0.0, 1.0, 40)
os.makedirs('fig', exist_ok=True)

fig, axes = plt.subplots(2, 4, figsize=(24, 12))

poisson_lams = [2.0, 3.0, 5.0, 8.0]
print("\n========== First row: Poisson Networks by mean degree ==========")
for idx, lam in enumerate(poisson_lams):
    ax = axes[0, idx]
    topologies = ['Uncorrelated Poisson', 'Correlated Poisson']

    print(f"--- Panel A-{idx + 1}: lambda = {lam} ---")
    for topo in topologies:
        print(f"  Calculating {topo}...")
        q_in, q_out, m_in, m_out = generate_sequences_from_c(topo, lam=lam, gamma=3.5)

        delta_pcs = []
        for theta in thetas:
            pc_dir, pc_undir = find_thresholds(theta, q_in, q_out, m_in, m_out)
            if not np.isnan(pc_dir) and not np.isnan(pc_undir):
                delta_pcs.append(pc_dir - pc_undir)
            else:
                delta_pcs.append(np.nan)

        color = 'tab:blue' if 'Uncorrelated' in topo else 'tab:red'
        marker = '^' if 'Uncorrelated' in topo else 'o'
        ax.plot(thetas, delta_pcs, label=f"{topo} ($\\lambda={lam}$)", color=color, linewidth=2, marker=marker, markersize=5)

    ax.axhline(0, color='black', linestyle='--', linewidth=1.5)
    ax.set_title(f'Poisson ($\\lambda={lam}$)', fontsize=16, fontweight='bold')
    ax.set_xlabel(r'Anchor Node Fraction ($\theta$)', fontsize=14)
    if idx == 0:
        ax.set_ylabel(r'Fragility Gap $\Delta p_c$', fontsize=16)
        ax.annotate('Poisson Networks', xy=(-0.25, 0.5), xycoords='axes fraction',
                    fontsize=20, fontweight='bold', ha='center', va='center', rotation=90)

    ax.set_ylim(-0.02, 0.35)
    ax.legend(fontsize=12, loc='best')
    ax.grid(True, linestyle=':', alpha=0.7)

sf_gammas = [2.5, 3.2, 3.9, 4.5]
print("\n========== Second row: Scale-Free Networks by gamma ==========")
for idx, gamma in enumerate(sf_gammas):
    ax = axes[1, idx]
    topologies = ['Uncorrelated SF', 'Correlated SF']

    print(f"--- Panel B-{idx + 1}: gamma = {gamma} ---")

    for topo in topologies:
        print(f"  Calculating {topo}...")
        q_in, q_out, m_in, m_out = generate_sequences_from_c(topo, lam=3.0, gamma=gamma)

        delta_pcs = []
        for theta in thetas:
            pc_dir, pc_undir = find_thresholds(theta, q_in, q_out, m_in, m_out)
            if not np.isnan(pc_dir) and not np.isnan(pc_undir):
                delta_pcs.append(pc_dir - pc_undir)
            else:
                delta_pcs.append(np.nan)

        color = 'tab:blue' if 'Uncorrelated' in topo else 'tab:red'
        marker = '^' if 'Uncorrelated' in topo else 'o'
        ax.plot(thetas, delta_pcs, label=f"{topo} ($\\gamma={gamma}$)", color=color, linewidth=2, marker=marker, markersize=5)

    ax.axhline(0, color='black', linestyle='--', linewidth=1.5)
    ax.set_title(f'Scale-Free ($\\gamma={gamma}$)', fontsize=16, fontweight='bold')
    ax.set_xlabel(r'Anchor Node Fraction ($\theta$)', fontsize=14)
    if idx == 0:
        ax.set_ylabel(r'Fragility Gap $\Delta p_c$', fontsize=16)
        ax.annotate('Scale-Free Networks', xy=(-0.25, 0.5), xycoords='axes fraction',
                    fontsize=20, fontweight='bold', ha='center', va='center', rotation=90)

    ax.set_ylim(-0.02, 0.75)
    ax.legend(fontsize=12, loc='best')
    ax.grid(True, linestyle=':', alpha=0.7)

plt.tight_layout()
plt.subplots_adjust(wspace=0.1, hspace=0.25, left=0.08)

output_filename = 'fig/Figure2(1)_delta_pc.pdf'
plt.savefig(output_filename, bbox_inches='tight', dpi=600)
print(f"\nFigure saved: {output_filename}")

plt.show()
